# Análisis de Comportamiento de Consumidores: Sector Electrónica
**Objetivos**: Limpieza, transformación y análisis exploratorio de un dataset simulado de e-commerce para identificar patrones de compra, estacionalidad y tasas de devolución en productos del rubro tecnológico.  

In [2]:
import pandas as pd


In [3]:
df_principal = pd.read_csv("Ecommerce_Consumer_Behavior_Analysis_Data.csv")

df_principal.head()

,Customer_ID,Age,Gender,Income_Level,Marital_Status,Education_Level,Occupation,Location,Purchase_Category,Purchase_Amount,...,Customer_Satisfaction,Engagement_with_Ads,Device_Used_for_Shopping,Payment_Method,Time_of_Purchase,Discount_Used,Customer_Loyalty_Program_Member,Purchase_Intent,Shipping_Preference,Time_to_Decision
0,37-611-6911,22,Female,Middle,Married,Bachelor's,Middle,Évry,Gardening & Outdoors,$333.80,...,7,NaN,Tablet,Credit Card,3/1/2024,True,False,Need-based,No Preference,2
1,29-392-9296,49,Male,High,Married,High School,High,Huocheng,Food & Beverages,$222.22,...,5,High,Tablet,PayPal,4/16/2024,True,False,Wants-based,Standard,6
2,84-649-5117,24,Female,Middle,Single,Master's,High,Huzhen,Office Supplies,$426.22,...,7,Low,Smartphone,Debit Card,3/15/2024,True,True,Impulsive,No Preference,3
3,48-980-6078,29,Female,Middle,Single,Master's,Middle,Wiwilí,Home Appliances,$101.31,...,1,NaN,Smartphone,Other,10/4/2024,True,True,Need-based,Express,10
4,91-170-9072,33,Female,Middle,Widowed,High School,Middle,Nara,Furniture,$211.70,...,10,NaN,Smartphone,Debit Card,1/30/2024,False,False,Wants-based,No Preference,4


In [4]:
df_recortado = df_principal[["Customer_ID", "Location", "Age", "Purchase_Amount", "Purchase_Category", "Time_of_Purchase", "Return_Rate"]].copy()

df_filtrado = df_recortado[ df_recortado["Purchase_Category"] == "Electronics"].copy()

df_filtrado.head()

,Customer_ID,Location,Age,Purchase_Amount,Purchase_Category,Time_of_Purchase,Return_Rate
19,67-159-7366,Monastyrshchina,29,$490.75,Electronics,7/8/2024,1
27,58-623-8404,Pasirhuni,50,$81.91,Electronics,7/24/2024,0
66,09-346-4992,Jesús Menéndez,46,$223.47,Electronics,9/20/2024,0
74,71-197-8227,Tocoa,37,$298.38,Electronics,4/27/2024,0
117,39-952-4459,Tessalit,18,$257.37,Electronics,2/1/2024,2


# Limpieza de Datos:
Convertimos la columna de precios de formato texto a numérico flotante. Se verificó previamente la ausencia de valores negativos para garantizar la integridad de las métricas.

In [ ]:
df_filtrado["Purchase_Amount"] = df_filtrado["Purchase_Amount"].str.replace("$", "").astype(float)

In [ ]:
compra_max = df_filtrado["Purchase_Amount"].max()

compra_min = df_filtrado["Purchase_Amount"].min()

compra_promedio = df_filtrado["Purchase_Amount"].mean()

compra_media = df_filtrado["Purchase_Amount"].median()

print(f"La compra más cara fue de ${compra_max:.2f}")
print(f"La compra más barata fue de ${compra_min:.2f}")
print(f"El promedio en el valor de compras es de ${compra_promedio:.2f}")
print(f"El valor medio de compras fue de ${compra_media:.2f}")

La compra más cara fue de $490.75
La compra más barata fue de $63.91
El promedio en el valor de compras es de $256.34
El valor medio de compras fue de $236.25


In [ ]:
dinero_total = df_filtrado["Purchase_Amount"].sum()

### 2. Segmentación por Rango Etario

In [10]:
# Asignamos rangos de edad y el nombre para cada rango
grupo_edad = [17, 25, 35, 50, 100]
nombre_grupo = ["18-25 años", "25-35 años", "35-50 años", "50-100 años"]

# Creamos la columna para el Rango de edad, en base al grupo y nombre ya establecidos
df_filtrado["Rango_Edad"] = pd.cut(df_filtrado["Age"], bins=grupo_edad, labels=nombre_grupo)

# Agrupamos los totales de compras por rango de edad
ranking_edad = df_filtrado.groupby("Rango_Edad", observed=False)[["Purchase_Amount"]].sum().reset_index()
# Hacemos que el orden sea descendiente
ranking_edad = ranking_edad.sort_values(by="Purchase_Amount", ascending=False)

# Calculamos porcentaje de compra en base a rango de edad
ranking_edad["Porcentaje_Edad"] = (ranking_edad["Purchase_Amount"] / dinero_total) * 100

display(ranking_edad.head(10))



,Rango_Edad,Purchase_Amount,Porcentaje_Edad
2,35-50 años,6679.05,48.250630
1,25-35 años,3959.43,28.603617
0,18-25 años,3203.93,23.145753
3,50-100 años,0.00,0.000000


### 3. Análisis de Estacionalidad de Ventas

In [11]:
df_filtrado["Time_of_Purchase"] = pd.to_datetime(df_filtrado["Time_of_Purchase"])

# Creamos una columna para el "Mes"
df_filtrado["Mes"] = df_filtrado["Time_of_Purchase"].dt.month

# Agrupamos los meses por estación
diccionario_estaciones = {
    1: "Verano", 2: "Verano", 3: "Otoño",
    4: "Otoño", 5: "Otoño", 6: "Invierno",
    7: "Invierno", 8: "Invierno", 9: "Primavera",
    10: "Primavera", 11: "Primavera", 12: "Verano"
}

# Creamos una columna para cada Estación
df_filtrado["Estación"] = df_filtrado["Mes"].map(diccionario_estaciones)

In [ ]:
# Agrupamos los totales de compras por estación del año
ranking_estacion = df_filtrado.groupby("Estación", observed=False)[["Purchase_Amount"]].sum().reset_index()
# Hacemos que el orden sea descendiente
ranking_estacion = ranking_estacion.sort_values(by="Estación", ascending=False)

# Calculamos porcentaje de compra en base a rango de edad
ranking_estacion["Porcentaje_Estación"] = (ranking_estacion["Purchase_Amount"] / dinero_total) * 100

ranking_estacion = ranking_estacion.sort_values(by="Purchase_Amount", ascending=False)

display(ranking_estacion)

,Estación,Purchase_Amount,Porcentaje_Estación
1,Otoño,4078.04,29.460477
0,Invierno,3850.80,27.818855
2,Primavera,3456.35,24.969279
3,Verano,2457.22,17.751389


### 4. Tasas de Devolución

In [ ]:
# Agrupamos por rango de edad y con el mediano del ratio de devolución
ranking_devoluciones = df_filtrado.groupby("Rango_Edad", observed=False)[["Return_Rate"]].mean().reset_index()

# Ordenamos de forma descendente desde "Return_Rate"
ranking_devoluciones = ranking_devoluciones.sort_values(by="Return_Rate", ascending=False)

display(ranking_devoluciones)

,Rango_Edad,Return_Rate
2,35-50 años,1.076923
1,25-35 años,0.937500
0,18-25 años,0.916667
3,50-100 años,NaN


### 5. Exportación del Modelo Limpio

In [33]:
df_filtrado.to_csv("Simulacro_Frávega.csv", index=False)